# Importing

In [1]:
import landbosse
import floris_cupy
import os
import cupy as cp
import numpy as np
import pandas as pd

from landbosse.main_function import run_landbosse # type: ignore

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import yaml
from scipy.stats import weibull_min
from shapely.geometry import Point, Polygon, LineString
from shapely.ops import unary_union
from floris import FlorisModel
from floris.wind_data import WindRose
import floris as _floris_pkg

# Path to FLORIS built-in default GCH config
_FLORIS_DEFAULT_YAML = str(
	pathlib.Path(_floris_pkg.__file__).parent / 'default_inputs.yaml'
)
print('FLORIS', _floris_pkg.__version__, '-- default yaml found:', _FLORIS_DEFAULT_YAML)
import numpy as np
from scipy.optimize import differential_evolution

from scipy.optimize import minimize
import numpy as np

FLORIS 4.6.4 -- default yaml found: /home/lavender/Studies/Design of Wind Farms/.venv/lib/python3.12/site-packages/floris/default_inputs.yaml


In [3]:
os.chdir(r"/home/lavender/Studies/Design of Wind Farms/Assignments/Assignment6/modular")

# Fixed Vars

In [4]:
d_ws = 0.17
d_ti = 12/100
d_fuel_cost = 9.5
d_line_freq = 50
d_standard_V = 220
d_interconnect_V = 100
d_rent = 15000
d_o_and_m = 0.012
d_discount = 3.6/1e2
d_life_time = 20
d_construction_time = 12		# Month

TURB_COST_PER_MW = 1.3 * 1e6

In [5]:
import numpy as np

rho = 1.225          # kg/m^3
D = 156.0            # m
V_rated = 9.0        # m/s (where power first reaches 3300 kW)
Ct_rated = 0.8027619595

A = np.pi * D**2 / 4
T_rated = 0.5 * rho * A * Ct_rated * V_rated**2

# Helper Funcs

In [6]:
lattitude_degree_to_km = 111
longitude_degree_to_km_at_55 = 63

In [7]:
def get_ref(stri):
	# This is using the DMS format - Degree Minutes Seconds
	
	lattLB = 55 + (14/60) + (39.2/(3600))
	longLB = 9 + (00/60) + (41.3/(3600))

	latt_deg = float(stri.strip().split("°")[0])
	latt_min = float(stri.strip().split("'")[0].split("°")[1])
	latt_sec = float(stri.strip().split("\"")[0].split("°")[1].split("'")[1].replace("\\", ""))

	long_deg = float(stri.strip().split("N")[1].split("°")[0])
	long_min = float(stri.strip().split("N")[1].split("'")[0].split("°")[1])
	long_sec = float(stri.strip().split("N")[1].split("\"")[0].split("°")[1].split("'")[1].replace("\\", ""))

	latt = latt_deg + (latt_min/60) + (latt_sec/(3600))
	long = long_deg + (long_min/60) + (long_sec/(3600))
 
	latt = (latt - lattLB) * lattitude_degree_to_km
 
	long = (long - longLB) * longitude_degree_to_km_at_55

	return latt, long

In [8]:
def get_ref_WGS84(latt, long):
	# This is using the WGS84 lattitude/longitude form.
	
	lattLB = 55 + (14/60) + (39.2/(3600))
	longLB = 9 + (00/60) + (41.3/(3600))
 
	latt = (latt - lattLB) * lattitude_degree_to_km * 1000
 
	long = (long - longLB) * longitude_degree_to_km_at_55 * 1000

	return latt, long

# Extracting Positions

In [9]:
db = r"optimizedLayout/BAR_BAU_IEA_3.3MW_16_layout.txt"
db = r"/home/lavender/Studies/Design of Wind Farms/Assignments/Assignment6/modular/optimizedLayout/BAR_BAU_IEA_3.3MW_18_layout.txt"

dir = r"optimizedLayout"

In [10]:
def linetoxy(string):
	string=string.replace("x=", "").replace("y=", "").replace(" m", "")
	return [float(i) for i in string.replace(",", "").split(":")[1].strip().split()]

substation_lines = []
turbine_lines = []

with open(db) as f:
	line = f.readline().strip()
	while line is not None:
		if "Substation" in line:
			line = f.readline().strip()
			substation_lines.append(linetoxy(line))
		if "Turbine" in line and "Positions" not in line:
			turbine_lines.append(linetoxy(line))
   
		line = f.readline().strip()
		
		if "Start" in line:
			break
 
# turbine_lines

# Landbosse

## Calcing Interconnect Distance

In [11]:
pl1 = get_ref_WGS84(55.26958668425792, 9.194729595538924)
pl2 = get_ref_WGS84(55.24081570901905, 9.208746443354379)

pl1,pl2, substation_lines

((2815.455285963104, 11545.214518952183),
 (-378.12296555154745, 12428.275931325848),
 [[1183.3892, 2374.3629]])

In [12]:
def point_to_line_distance_meters(pl1, pl2, point):
	"""Shortest distance from a point to an infinite line through pl1 and pl2. Returns meters."""
	p1 = np.array(pl1, dtype=float)
	p2 = np.array(pl2, dtype=float)
	p  = np.array(point, dtype=float)
	
	# Vector from p1 to p2
	line_vec = p2 - p1
	# Vector from p1 to point
	point_vec = p - p1
	
	# Cross product magnitude in 2D: |x1*y2 - y1*x2|
	cross = abs(line_vec[0] * point_vec[1] - line_vec[1] * point_vec[0])
	line_len = np.linalg.norm(line_vec)
	
	return cross / line_len

METERS_TO_MILES = 0.000621371

# d_interconnect_distance

d_interconnect_distance = 6.34

for pt in substation_lines:
	d_m = point_to_line_distance_meters(pl1, pl2, pt)
	d_interconnect_distance = d_m * METERS_TO_MILES
	print(f"Point {pt}: {d_m:.3f} m  =  {d_interconnect_distance:.6f} miles")


Point [1183.3892, 2374.3629]: 9274.123 m  =  5.762671 miles


## Setting LandBOSSE

In [13]:
from pathlib import Path

project_list_path = Path("/home/lavender/Studies/Design of Wind Farms/modules/LandBOSSE/input/project_list.xlsx")
backup_path = project_list_path.with_name("project_list_backup.xlsx")
default_path = project_list_path.with_name(f"{project_list_path.stem}_default{project_list_path.suffix}")


old_projects = pd.read_excel(project_list_path)


old_projects = pd.DataFrame(old_projects.iloc[[0]])

old_projects

,Project ID,Project data file,Total project construction time (months),Turbine rating MW,Hub height m,Rotor diameter m,Turbine spacing (times rotor diameter),Row spacing (times rotor diameter),Number of turbines,Breakpoint between base and topping (percent),...,Allow same flag,Override total management cost for distributed (0 does not override),Markup contingency,Markup warranty management,Markup sales and use tax,Markup overhead,Markup profit margin,Utility Interconnection Fees (Small DW only),Labor cost multiplier,Crane breakdown fraction
0,Assignment 6,iea36_120_project_data,12,3.3,120,156,5,0,16,0,...,n,1519250,0.03,0.0002,0,0.05,0.05,0,1,0


In [14]:
projects = old_projects.copy()

projects["Project ID"] = ["Assignment 6"]
projects["Project data file"] = "iea36_120_project_data"
projects["Total project construction time (months)"] = d_construction_time
projects["Hub height m"] = 120
projects["Rotor diameter m"] = 156
projects["Turbine spacing (times rotor diameter)"] = 5
projects["Line Frequency (Hz)"] = 50
projects["Number of turbines"] = 16
projects["Fuel cost USD per gal"] = 9.5
projects["Wind shear exponent"] = 0.17
projects["Turbine rating MW"] = 3.3
projects["Row spacing (times rotor diameter)"] = 0
projects["Rated Thrust (N)"] = T_rated
projects["Distance to interconnect (miles)"] = d_interconnect_distance
projects["Interconnect Voltage (kV)"] = 132

# Write the updated projects DataFrame
projects.to_excel(project_list_path, index=False)

projects

,Project ID,Project data file,Total project construction time (months),Turbine rating MW,Hub height m,Rotor diameter m,Turbine spacing (times rotor diameter),Row spacing (times rotor diameter),Number of turbines,Breakpoint between base and topping (percent),...,Allow same flag,Override total management cost for distributed (0 does not override),Markup contingency,Markup warranty management,Markup sales and use tax,Markup overhead,Markup profit margin,Utility Interconnection Fees (Small DW only),Labor cost multiplier,Crane breakdown fraction
0,Assignment 6,iea36_120_project_data,12,3.3,120,156,5,0,16,0,...,n,1519250,0.03,0.0002,0,0.05,0.05,0,1,0


## Running LandBOSSE

In [15]:
WriteExcel = False
Display = True

# Run
landbosse_results = run_landbosse(np.array(turbine_lines), np.array(substation_lines), 45, WriteExcel, Display)

>>>>>>>> Begin run 2026-06-28 14:49:30.876661 <<<<<<<<<<
>>> Project and parametric lists loaded
<><><><><><><><><><><><><><><><><><> Assignment 6 <><><><><><><><><><><><><><><><><><>
>>> project_id: Assignment 6
>>> Project data: input/project_data/iea36_120_project_data.xlsx


In [16]:
landbosse_results

,Project ID with serial,Number of turbines,Turbine rating MW,Rotor diameter m,Module,Type of cost,Cost per turbine,Cost per project,Cost per kW
0,Assignment 6,18,3.3,156,FoundationCost,Equipment rental,4951.840212,8.913312e+04,1.500558
1,Assignment 6,18,3.3,156,FoundationCost,Labor,101575.594241,1.828361e+06,30.780483
2,Assignment 6,18,3.3,156,FoundationCost,Materials,91005.905941,1.638106e+06,27.577547
3,Assignment 6,18,3.3,156,FoundationCost,Mobilization,9876.667020,1.777800e+05,2.992929
4,Assignment 6,18,3.3,156,SitePreparationCost,Materials,4317.869937,7.772166e+04,1.308445
5,Assignment 6,18,3.3,156,SitePreparationCost,Equipment rental,1635.273527,2.943492e+04,0.495537
6,Assignment 6,18,3.3,156,SitePreparationCost,Labor,10341.098088,1.861398e+05,3.133666
7,Assignment 6,18,3.3,156,SitePreparationCost,Other,58930.022222,1.060740e+06,17.857582
8,Assignment 6,18,3.3,156,SitePreparationCost,Mobilization,3761.213189,6.770184e+04,1.139762
9,Assignment 6,18,3.3,156,SubstationCost,Other,211513.994748,3.807252e+06,64.095150


# Calcing AEP

## Weibull Distro

In [17]:
binsize = 3 # m/s

WD_BINS  = np.array([0,30,60,90,120,150,180,210,240,270,300,330], dtype=float)
WB_SCALE = np.array([9.785,8.284,8.721,9.633,10.114,8.340,
					  8.936,10.759,11.710,11.363,10.682,8.965])  # lambda [m/s]
WB_SHAPE = np.array([2.306,2.089,1.888,1.935,1.945,1.902,
					  1.909,1.910,1.968,2.049,2.064,1.928])       # k [-]
FREQ_WD  = np.array([14.71,6.09,6.16,8.17,9.58,6.05,
					  5.34,7.27,8.00,14.60,7.78,6.25]) / 100.0

# Wind-speed bins: cut-in to cut-out at 1 m/s resolution
WS_BINS   = np.arange(3.0, 26.0, 1.0)   # 23 bins
SITE1_TI  = d_ti      # turbulence intensity (Table 1)
WIND_SHEAR= d_ws      # shear exponent alpha (Table 1)

# Build joint freq table P(WD_i, WS_j) from per-sector Weibull distributions
freq_table = np.zeros((len(WD_BINS), len(WS_BINS)))
for i, (k, lam) in enumerate(zip(WB_SHAPE, WB_SCALE)):
	p_ws = weibull_min.pdf(WS_BINS, c=k, scale=lam) * binsize
	p_ws /= p_ws.sum()   # renormalise (remove truncated tail)
	freq_table[i, :] = FREQ_WD[i] * p_ws

wind_rose = WindRose(
	wind_directions=WD_BINS,
	wind_speeds=WS_BINS,
	ti_table=SITE1_TI,
	freq_table=freq_table,
)
print(f'WindRose: {len(WD_BINS)} dirs x {len(WS_BINS)} speeds')
print(f'Frequency sum: {freq_table.sum():.6f}  (should be 1.0)')

WindRose: 12 dirs x 23 speeds
Frequency sum: 1.000000  (should be 1.0)


## WindRose Seeing

In [18]:
# wind_rose.plot()

## Running Floris

In [19]:
x = [i[0] for i in turbine_lines]
y = [i[1] for i in turbine_lines]

In [20]:
fmodel = FlorisModel(r"inputs/gch.yaml")
fmodel.set(layout_x=x,layout_y=y,wind_data=wind_rose,wind_shear=d_ws)
fmodel.run()

fmodel_no_wake = FlorisModel(r"inputs/gch.yaml")
fmodel_no_wake.set(layout_x=x,layout_y=y,wind_data=wind_rose,wind_shear=d_ws)
fmodel_no_wake.run_no_wake()

/home/lavender/Studies/Design of Wind Farms/.venv/lib/python3.12/site-packages/floris/core/flow_field.py:169: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  * np.power(


## Calcing Wake Loss

In [21]:
AEP_GWH = fmodel.get_farm_AEP()/1e9
AEP_GWH_NO_WAKE = fmodel_no_wake.get_farm_AEP()/1e9
WAKE_LOSSES = 100 * (AEP_GWH_NO_WAKE - AEP_GWH)/AEP_GWH_NO_WAKE
FARM_EFFICIENCY = 100 * (AEP_GWH)/AEP_GWH_NO_WAKE

print(f"AEP in GWh: {AEP_GWH: .3f} GWh")
print(f"AEP with no wake in GWh: {AEP_GWH_NO_WAKE: .3f} GWh")
print(f"Farm Efficiency: {FARM_EFFICIENCY: .3f} %")
print(f"Wake Losses: {WAKE_LOSSES: .3f} %")

AEP in GWh:  309.152 GWh
AEP with no wake in GWh:  346.687 GWh
Farm Efficiency:  89.173 %
Wake Losses:  10.827 %


## Calcing Capacity Factors

In [22]:
MAX_CAPACITY_YEAR = (len(turbine_lines)* 3300 * 8760/ 1e6)

CAPACITY_FACTOR = 100 *AEP_GWH/(len(turbine_lines)* 3300 * 8760/ 1e6)

print(f"Max Capacity of Wind Farm in a Year: {MAX_CAPACITY_YEAR: .3f} GWh")
print(f"Power Produced by Wind Farm in a Year: {AEP_GWH: .3f} GWh")
print(f"Capacity Factor: {CAPACITY_FACTOR: .3f} %")

Max Capacity of Wind Farm in a Year:  520.344 GWh
Power Produced by Wind Farm in a Year:  309.152 GWh
Capacity Factor:  59.413 %


## BoP Cost Calcing

In [23]:
bop_cost = landbosse_results['Cost per project'].sum()

print(f"BoP Cost:               ${bop_cost:,.0f}")

bop_by_module = landbosse_results.groupby('Module')['Cost per project'].sum().sort_values(ascending=False)
print("\nBoP breakdown by module:")
print(bop_by_module.apply(lambda x: f"  ${x:,.0f}").to_string())

BoP Cost:               $21,047,305

BoP breakdown by module:
Module
TransportCost            $5,900,000
SubstationCost           $3,807,252
FoundationCost           $3,733,380
ErectionCost             $2,300,861
GridConnectionCost       $1,786,682
ManagementCost           $1,519,250
SitePreparationCost      $1,421,739
CollectionCost             $428,141
DevelopmentCost            $150,000


## Calcing Investments

In [24]:
AEP_MWh = float(AEP_GWH) * 1e3  # convert GWh to MWh

INSTALL_CAPACITY = len(turbine_lines) * 3.30

turbine_cost = TURB_COST_PER_MW * INSTALL_CAPACITY
I = bop_cost + turbine_cost

print(f"BoP Cost:           ${bop_cost:,.0f}")
print(f"Turbine Cost:       ${turbine_cost:,.0f}")
print(f"Total Investment I: ${I:,.0f}")

BoP Cost:           $21,047,305
Turbine Cost:       $77,220,000
Total Investment I: $98,267,305


## Energy-weighted mean wind speed for spot price

In [25]:
# Convert cupy arrays to numpy
farm_power = cp.asnumpy(fmodel.get_farm_power())   # [W] per FLORIS condition

# Build matching WS grid (must match WindRose condition order)
wd_mesh, ws_mesh = np.meshgrid(WD_BINS, WS_BINS, indexing='ij')
freq_flat = freq_table.flatten()                     # numpy
ws_flat = ws_mesh.flatten()                          # numpy

farm_power = cp.asnumpy(fmodel.get_farm_power()).flatten()   # shape (276,)

# Energy per condition [Wh] = power × frequency × 8760 h
energy_per_cond = farm_power * freq_flat * 8760.0

# Energy-weighted mean wind speed
ws_energy_weighted = np.sum(energy_per_cond * ws_flat) / np.sum(energy_per_cond)
print(f"\nEnergy-weighted mean WS: {ws_energy_weighted:.2f} m/s")


Energy-weighted mean WS: 11.77 m/s


## Yearly spot prices & revenue

In [26]:
years = np.arange(2021, 2041)

# Denmark average spot prices A ($/MWh) from Table 4
A_denmark = np.array([
	100.5, 249.8, 154.3, 74.3, 57.7,   # 2021-2025
	53.6,  63.1,  51.2,  38.9, 35.9,   # 2026-2030
] + [32.6] * 10)                        # 2031-2040

SPOT_C = 1.235
SPOT_K = 0.04295

# Effective spot price for each year [$/MWh]
effective_spot = A_denmark * (SPOT_C - SPOT_K * ws_energy_weighted)

# Annual revenue [$]
annual_revenue = AEP_MWh * effective_spot

# Annual O&M: $0.012/kWh = $12/MWh
annual_om = AEP_MWh * 12.0

# Annual land rental: $15,000/MW-yr (Denmark)
annual_rental = INSTALL_CAPACITY * d_rent

# Net cash flow years 1-20
annual_net = annual_revenue - annual_om - annual_rental

## PI & NPV

In [27]:
pv_net = np.sum(annual_net / (1 + d_discount)**np.arange(1, d_life_time + 1))
PI = pv_net / I
NPV = pv_net - I

print(f"\n--- Results ---")
print(f"Annual Revenue (avg): ${annual_revenue.mean():,.0f}")
print(f"Annual O&M:           ${annual_om:,.0f}")
print(f"Annual Rental:        ${annual_rental:,.0f}")
print(f"Annual Net (avg):     ${annual_net.mean():,.0f}")
print(f"PV of net cash flows: ${pv_net:,.0f}")
print(f"NPV:                  ${NPV:,.0f}")
print(f"PI:                   {PI:.3f}")


--- Results ---
Annual Revenue (avg): $13,587,281
Annual O&M:           $3,709,824
Annual Rental:        $891,000
Annual Net (avg):     $8,986,456
PV of net cash flows: $150,615,731
NPV:                  $52,348,425
PI:                   1.533


## Internal Return Rate

In [28]:
from scipy.optimize import newton

def npv_of_rate(r):
	"""NPV as a function of discount rate r (scalar)."""
	return -I + np.sum(annual_net / (1 + r)**np.arange(1, d_life_time + 1))

# Solve for the rate where NPV = 0
# Newton-Raphson starting from the user's discount rate
IRR = newton(npv_of_rate, x0=d_discount, tol=1e-12, maxiter=200)

# print(f"\n--- IRR ---")
print(f"IRR: {IRR*100:.3f} %")
# print(f"Discount rate used: {d_discount*100:.3f} %")
# print(f"Spread (IRR - r): {(IRR - d_discount)*100:.3f} %")

IRR: 15.740 %


## LCOE

In [29]:
annuity = ((1 + d_discount)**d_life_time - 1) / (d_discount * (1 + d_discount)**d_life_time)
annual_fixed_cost = annual_om + annual_rental
TLCC = I + annual_fixed_cost * annuity
LCOE = TLCC / (AEP_MWh * annuity)

print(f"\nLCOE:                 ${LCOE:.2f}/MWh")
print(f"Annuity factor:       {annuity:.4f}")


LCOE:                 $37.45/MWh
Annuity factor:       14.0847


## Wake Steering

In [ ]:
# # ------------------------------------------------------------------
# #  Parallel wake-steering model setup
# # ------------------------------------------------------------------
from floris_cupy import ParFlorisModel  # try this first
# If the above fails, use: from floris import ParFlorisModel

from floris_cupy.optimization.yaw_optimization.yaw_optimizer_sr import YawOptimizationSR

# Create parallel model (same API as FlorisModel)
fmodel_wake_steering = ParFlorisModel(
	r"inputs/gch.yaml",
	max_workers=16,              # CPU processes
	n_wind_condition_splits=4   # split wind conditions across workers
)

fmodel_wake_steering.set(
	layout_x=x,
	layout_y=y,
	wind_data=wind_rose,
	wind_shear=d_ws
)

In [32]:
fmodel_wake_steering = FlorisModel(r"inputs/gch.yaml")
fmodel_wake_steering.set(
	layout_x=x,
	layout_y=y,
	wind_data=wind_rose,
	wind_shear=d_ws
)

n_turbines = len(turbine_lines)
n_findex   = len(WD_BINS) * len(WS_BINS)

In [37]:
from floris_cupy.optimization.yaw_optimization.yaw_optimizer_pr import YawOptimizationPR # pyright: ignore[reportMissingImports]

yaw_opt = YawOptimizationPR(fmodel_wake_steering, display=True)
df_opt  = yaw_opt.optimize()

# Stack the per-row arrays into a proper 2D numpy array (n_findex, n_turbines)
yaw_optimal = np.stack(df_opt["yaw_angles_opt"].values)   # shape: (276, 16)

fmodel_wake_steering.set(yaw_angles=yaw_optimal)
fmodel_wake_steering.run()

print("Yaw optimization complete.")

INFO: Baseline yaw angles were not specified and were derived from the floris object.
INFO: The inherent yaw angles in the floris object are not all 0.0 degrees.


/home/lavender/Studies/Design of Wind Farms/.venv/lib/python3.12/site-packages/floris/core/flow_field.py:169: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  * np.power(


[Parallel Refine] Processing pass=0, turbine_depth=0 (0.0%)
[Parallel Refine] Processing pass=0, turbine_depth=1 (2.8%)
[Parallel Refine] Processing pass=0, turbine_depth=2 (5.6%)
[Parallel Refine] Processing pass=0, turbine_depth=3 (8.3%)
[Parallel Refine] Processing pass=0, turbine_depth=4 (11.1%)
[Parallel Refine] Processing pass=0, turbine_depth=5 (13.9%)
[Parallel Refine] Processing pass=0, turbine_depth=6 (16.7%)
[Parallel Refine] Processing pass=0, turbine_depth=7 (19.4%)
[Parallel Refine] Processing pass=0, turbine_depth=8 (22.2%)
[Parallel Refine] Processing pass=0, turbine_depth=9 (25.0%)
[Parallel Refine] Processing pass=0, turbine_depth=10 (27.8%)
[Parallel Refine] Processing pass=0, turbine_depth=11 (30.6%)
[Parallel Refine] Processing pass=0, turbine_depth=12 (33.3%)
[Parallel Refine] Processing pass=0, turbine_depth=13 (36.1%)
[Parallel Refine] Processing pass=0, turbine_depth=14 (38.9%)
[Parallel Refine] Processing pass=0, turbine_depth=15 (41.7%)
[Parallel Refine] Proc

In [34]:

AEP_GWH_WAKE_STEERING = fmodel_wake_steering.get_farm_AEP()/1e9

print(f"Wake Steering Improvement: {(100*(AEP_GWH_WAKE_STEERING-AEP_GWH)/AEP_GWH):.3f} %")
print(f"Wake Steering Improvement in GWh: {(AEP_GWH_WAKE_STEERING-AEP_GWH):.3f} GWh")

Wake Steering Improvement: 1.027 %
Wake Steering Improvement in GWh: 3.176 GWh


In [35]:
# fmodel_wake_steering = FlorisModel(r"inputs/gch.yaml")
# fmodel_wake_steering.set(
#     layout_x=x,
#     layout_y=y,
#     wind_data=wind_rose,
#     wind_shear=d_ws
# )

# n_turbines = len(turbine_lines)
# n_findex   = len(WD_BINS) * len(WS_BINS)

# # ------------------------------------------------------------------
# #  Yaw optimization  (only run this if you want OPTIMAL wake steering)
# # ------------------------------------------------------------------
# # try:
# # floris_cupy may mirror the floris v4 API
# from floris_cupy.optimization.yaw_optimization.yaw_optimizer_sr import YawOptimizationSR

# yaw_opt = YawOptimizationSR(fmodel_wake_steering)
# df_opt  = yaw_opt.optimize()          # returns DataFrame with optimal yaw angles

# # Apply the optimized yaw angles back to the model
# yaw_optimal = df_opt["yaw_angles_opt"].values.reshape(n_findex, n_turbines)
# fmodel_wake_steering.set(yaw_angles=yaw_optimal)
# fmodel_wake_steering.run()

# print("Yaw optimization complete.")
# # except Exception as e:
# #     print(f"Yaw optimization not available or failed: {e}")

# Calcing PI and LCoE